# Лабораторная работа №4. Коммутатор учится

В Lab 003 мы нашли **destination MAC в Ethernet-кадре**. Но откуда реальный switch знает, в какой порт отправить этот кадр? Именно это мы сейчас выясним.

Проверим два состояния одного Linux bridge: **destination MAC ещё неизвестен → кадр flood-ится**, затем **source MAC выучен → следующий кадр идёт только на нужный порт**. После этого сломаем ровно один порт и локализуем отказ.

## Три хоста, один настоящий L2-коммутатор

![Трёхпортовая топология](assets/topology.svg)

`sw1` — Linux bridge `br0` с тремя портами: `eth1 → pc1`, `eth2 → pc2`, `eth3 → pc3`. Третий хост нужен не для маршрутизации, а как **наблюдатель**: если switch разошлёт чужой unicast на все порты, он увидит этот кадр.

MAC заранее известны: `pc1 …04:01`, `pc2 …04:02`, `pc3 …04:03`. Полный формат `02:00:00:00:04:XX`. Management `eth0` не трогаем.

### Q1. Пока bridge не получал кадров, знает ли он автоматически, где находится MAC `02:00:00:00:04:02`? По какому полю **входящего** кадра switch может начать строить таблицу соответствия MAC → port?

`введите ваш ответ`

<details>
<summary>Проверить ответ с ИИ (необязательно)</summary>

**AI-REVIEW-PROMPT / REVIEW-ONLY**

Проверь мой ответ на вопрос Q1, который находится выше.  
Не отвечай на исходный вопрос и не выдавай эталонный ответ.  
Оцени логику, отметь пропущенный шаг, задай 1–2 наводящих вопроса.  
Не переписывай мой ответ и не давай готовую CLI-команду для следующего шага.

</details>

## 1. Сначала исследуем switch

Откройте terminal `sw1` и посмотрите:

```bash
bridge link
bridge fdb show br br0 dynamic
```

На трёх портах должен быть `master br0`. В FDB нас интересуют именно **dynamic записи**: постоянные служебные/local MAC bridge к результатам обучения хостов не относятся.

Теперь подготовим только учебный data plane. Адреса нужны для запуска `ping`; сам switch по ним не принимает решений.

### Подготовьте хосты

На `pc1`:

```bash
ip link set eth1 up
ip -4 addr flush dev eth1
ip addr add 10.4.0.10/24 dev eth1
ip neigh replace 10.4.0.20 lladdr 02:00:00:00:04:02 nud permanent dev eth1
ip neigh replace 10.4.0.30 lladdr 02:00:00:00:04:03 nud permanent dev eth1
```

На `pc2`:

```bash
ip link set eth1 up
ip -4 addr flush dev eth1
ip addr add 10.4.0.20/24 dev eth1
ip neigh replace 10.4.0.10 lladdr 02:00:00:00:04:01 nud permanent dev eth1
```

На `pc3`:

```bash
ip link set eth1 up
ip -4 addr flush dev eth1
ip addr add 10.4.0.30/24 dev eth1
ip neigh replace 10.4.0.10 lladdr 02:00:00:00:04:01 nud permanent dev eth1
ip link set eth1 promisc on
```

Статические neighbor-записи здесь **не являются MAC-таблицей switch**. Мы устранили ARP-обмен между хостами, чтобы первый наблюдаемый кадр был unicast для `pc2`, а не ARP broadcast. ARP как отдельный механизм будет в Lab 005. Режим `promisc` помогает `pc3` видеть кадры с чужим destination MAC.

### Q2. `pc1` знает MAC `pc2` из своей neighbor table, но `sw1` пока не знает, за каким портом находится MAC `pc2`. Если после очистки FDB `pc1` пошлёт unicast к `pc2`, попадёт ли его копия на порт `pc3`? Почему?

`введите ваш ответ`

<details>
<summary>Проверить ответ с ИИ (необязательно)</summary>

**AI-REVIEW-PROMPT / REVIEW-ONLY**

Проверь мой ответ на вопрос Q2, который находится выше.  
Не отвечай на исходный вопрос и не выдавай эталонный ответ.  
Оцени логику, отметь пропущенный шаг, задай 1–2 наводящих вопроса.  
Не переписывай мой ответ и не давай готовую CLI-команду для следующего шага.

</details>

## 2. Первый кадр: destination неизвестен switch

На `sw1` очистите **только dynamic FDB**, не меняя состав bridge:

```bash
bridge fdb flush dev eth1 master dynamic
bridge fdb flush dev eth2 master dynamic
bridge fdb flush dev eth3 master dynamic
bridge fdb show br br0 dynamic
```

Теперь запустите следующую capture-cell на `pc3:eth1`. Пока она работает, **один раз** выполните в terminal `pc1`:

```bash
ping -c 1 -W 1 10.4.0.20
```

Пока capture идёт, не запускайте другие ping. Так первый Echo Request останется наиболее понятным кадром для анализа.

In [ ]:
%%capture_traffic pc3:eth1 --filter "icmp" --timeout 12 --packets 50 --save captures/unknown-unicast-pc3.pcap
import time
print("Пока pc3 слушает, запустите на pc1: ping -c 1 -W 1 10.4.0.20")
time.sleep(10)


### Что увидел третий порт?

Постройте статическую таблицу реальных кадров с `pc3`. Если кадр для `pc2` есть в capture `pc3`, это и есть наблюдаемое unknown-unicast flooding.

In [ ]:
from cms_labs_jupyter.traffic_capture import PcapViewer
from IPython.display import display

viewer_unknown = PcapViewer("captures/unknown-unicast-pc3.pcap", "icmp", limit=30)
unknown = viewer_unknown.packets()
display(unknown)
if not unknown.empty:
    number = int(unknown.iloc[0]["№"])
    print(f"\nFrame {number} · protocol tree (pc3)\n")
    print("\n".join(viewer_unknown.packet(number, "protocols").splitlines()[:65]))
else:
    print("Нет ICMP-кадров: проверьте promisc на pc3, очистите dynamic FDB и повторите ping внутри capture.")


### Посмотрите, что выучил switch

На `sw1`:

```bash
bridge fdb show br br0 dynamic
```

После первого Echo Request bridge может выучить `pc1 → eth1`, а после Echo Reply — `pc2 → eth2`. Источник знания — **source MAC входящего кадра**. `destination MAC` служит для выбора выходного порта и не означает, что адресат уже выучен.

## Как flooding превращается в точную пересылку

![Source learning, flooding и known unicast](assets/mac-learning.svg)

В анимации три фазы: `UNKNOWN → SOURCE LEARNED → KNOWN UNICAST`. Сравнивайте их с таблицей, которую только что получили на настоящем bridge. Обратите внимание, что switch учится по **входящим** кадрам, а не по содержимому ICMP.

### Q3. Теперь FDB знает `MAC pc2 → eth2`. Что вы ожидаете увидеть в новом capture на `pc3:eth1`, если `pc1` ещё раз отправит ICMP к `pc2`? Изменился ли при этом сам destination MAC кадра?

`введите ваш ответ`

<details>
<summary>Проверить ответ с ИИ (необязательно)</summary>

**AI-REVIEW-PROMPT / REVIEW-ONLY**

Проверь мой ответ на вопрос Q3, который находится выше.  
Не отвечай на исходный вопрос и не выдавай эталонный ответ.  
Оцени логику, отметь пропущенный шаг, задай 1–2 наводящих вопроса.  
Не переписывай мой ответ и не давай готовую CLI-команду для следующего шага.

</details>

## 3. Второй кадр: destination уже известен

Не очищайте FDB. Запустите capture на `pc3:eth1` ещё раз, и во время него на `pc1`:

```bash
ping -c 2 -W 1 10.4.0.20
```

Теперь unicast для `pc2` должен уйти на `sw1:eth2`, не на `eth3`. Поэтому **пустой ICMP-capture на наблюдателе — ожидаемый результат**. Если вы видите старые кадры, убедитесь, что это новый `.pcap`.

In [ ]:
%%capture_traffic pc3:eth1 --filter "icmp" --timeout 12 --packets 50 --save captures/known-unicast-pc3.pcap
import time
print("Повторите на pc1: ping -c 2 -W 1 10.4.0.20")
time.sleep(10)


In [ ]:
from cms_labs_jupyter.traffic_capture import PcapViewer
from IPython.display import display

try:
    viewer_known = PcapViewer("captures/known-unicast-pc3.pcap", "icmp", limit=30)
    known = viewer_known.packets()
    display(known)
    print("ICMP на pc3 после обучения FDB:", len(known), "кадров")
except Exception as exc:
    print("Не удалось прочитать PCAP. Это НЕ доказательство отсутствия трафика; проверьте capture и повторите опыт:", exc)


### Откуда переключение flooding → forwarding?

Первые кадры пришлось flood-ить, поскольку destination MAC ещё не был в FDB. Ответ `pc2` научил bridge соответствию `MAC pc2 → eth2`. Теперь switch выбирает конкретный выходной порт и не рассылает тот же unicast наблюдателю `pc3`.

Пустая таблица capture имеет смысл только вместе с **успешным ping** и **правильной FDB**. Если ping не прошёл, отсутствие кадров на `pc3` само по себе ничего не доказывает.

## 4. Первый CHECK: switch работает

Чтобы убедиться, что bridge выучил и `pc3`, отправьте **один** кадр от него: в terminal `pc3` выполните `ping -c 1 -W 1 10.4.0.10`.

Теперь на `sw1`:

```bash
bridge link
bridge fdb show br br0 dynamic
```

Ожидаем: `MAC pc1 → eth1`, `MAC pc2 → eth2`, `MAC pc3 → eth3`. Запустите **CHECK**. Он проверяет состав bridge, адреса hosts, две независимые связности и learned MAC-port entries. Кнопка CHECK не оценивает ответы в notebook или историю команд.

## 5. Controlled failure: отключаем только порт pc2 от bridge

Теперь намеренно сделаем проблему **внутри switch**. `pc2` по-прежнему имеет IPv4, `pc1` и `pc3` остаются подключены — изменится только членство `sw1:eth2` в `br0`.

### Q4. Если удалить `sw1:eth2` из bridge, какие связи должны продолжить работать (`pc1→pc2` или `pc1→pc3`)? Изменятся ли IPv4-адреса хостов? Какие признаки в CHECK помогут локализовать отказ?

`введите ваш ответ`

<details>
<summary>Проверить ответ с ИИ (необязательно)</summary>

**AI-REVIEW-PROMPT / REVIEW-ONLY**

Проверь мой ответ на вопрос Q4, который находится выше.  
Не отвечай на исходный вопрос и не выдавай эталонный ответ.  
Оцени логику, отметь пропущенный шаг, задай 1–2 наводящих вопроса.  
Не переписывай мой ответ и не давай готовую CLI-команду для следующего шага.

</details>

На `sw1`:

```bash
ip link set eth2 nomaster
bridge link
```

На `pc1`:

```bash
ping -c 2 -W 1 10.4.0.20
ping -c 2 -W 1 10.4.0.30
```

Запустите **CHECK**. Не исправляйте сразу: посмотрите, какой port-check и какой end-to-end тест стал красным, а что осталось зелёным.

### Q5. После поломки `pc1→pc3` работает, а `pc1→pc2` нет. Почему это указывает на путь через `sw1:eth2`, а не на сломанный `pc1:eth1` или неправильную подсеть у всех хостов?

`введите ваш ответ`

<details>
<summary>Проверить ответ с ИИ (необязательно)</summary>

**AI-REVIEW-PROMPT / REVIEW-ONLY**

Проверь мой ответ на вопрос Q5, который находится выше.  
Не отвечай на исходный вопрос и не выдавай эталонный ответ.  
Оцени логику, отметь пропущенный шаг, задай 1–2 наводящих вопроса.  
Не переписывай мой ответ и не давай готовую CLI-команду для следующего шага.

</details>

### Локализуем, не угадывая

На `sw1`:

```bash
bridge link
ip -o link show dev eth2
bridge fdb show br br0 dynamic
```

На `pc2`:

```bash
ip -br link show eth1
ip -br addr show eth1
```

У `pc2` его собственный `eth1` и адрес не исчезли. Но `sw1:eth2` больше не состоит в `br0`. Значит, неисправность не во всём L2-домене и не в IPv4-адресации, а в конкретном членстве порта switch.

## 6. Исправляем ровно то, что сломали

На `sw1`:

```bash
ip link set eth2 master br0
ip link set eth2 up
bridge link
```

На `pc1`:

```bash
ping -c 3 10.4.0.20
ping -c 2 10.4.0.30
```

На `pc3`: `ping -c 1 10.4.0.10`. Затем на `sw1` ещё раз проверьте `bridge fdb show br br0 dynamic`.

Запустите **FINAL CHECK**: связность и MAC learning должны восстановиться.

### Q6. Сформулируйте в 4–6 предложениях причинную модель: как switch узнаёт порт источника, что делает с unknown unicast, когда переходит к known unicast и почему изменение одного порта может нарушить только часть сети.

`введите ваш ответ`

<details>
<summary>Проверить ответ с ИИ (необязательно)</summary>

**AI-REVIEW-PROMPT / REVIEW-ONLY**

Проверь мой ответ на вопрос Q6, который находится выше.  
Не отвечай на исходный вопрос и не выдавай эталонный ответ.  
Оцени логику, отметь пропущенный шаг, задай 1–2 наводящих вопроса.  
Не переписывай мой ответ и не давай готовую CLI-команду для следующего шага.

</details>

## Итог Lab 004

- `bridge FDB` и `ip neigh` — **разные** таблицы: switch учится MAC → ingress port, host сопоставляет IP → MAC;
- switch читает source MAC для обучения и destination MAC для выбора egress;
- unknown-unicast flood идёт на остальные порты этого L2-домена;
- после ответа от pc2 dynamic FDB знает `MAC pc2 → eth2`, и следующие кадры туда пересылаются адресно;
- наблюдатель pc3 превращает flooding из картинки в измеримое событие;
- смена port membership ломает нужный сегмент, даже если у всех hosts правильные IP;
- CHECK проверяет реальный bridge state / reachability, а не написанные студентом ответы.

В Lab 005 вернёмся к автоматическому узнаванию MAC адресата — ARP.